<a href="https://colab.research.google.com/github/StrawEater/PracticasPDI3erBimestre/blob/main/Laboratorios/labo4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🌊 Laboratorio 4: De Álgebra Lineal a Fourier
### *Una deducción visual, intuitiva y rigurosa sin fórmulas caídas del cielo*

---

### 🎯 Objetivo de este laboratorio
Si cursaste **Álgebra Lineal básica**, ya sabés el 95% de lo que se necesita para entender Fourier:
1. Sabés qué es un vector $\vec{v} \in \mathbb{R}^n$.
2. Sabés calcular el **producto interno** (o producto escalar / dot product) entre dos vectores: $\langle \vec{u}, \vec{v} \rangle = \sum u_i v_i$.
3. Sabés qué significa que dos vectores sean **ortogonales** (su producto interno da $0$).
4. Sabés cómo proyectar un vector sobre una base ortonormal para encontrar sus **coordenadas**.
5. Sabés qué es una matriz de **cambio de base**.

En este laboratorio vamos a demostrar que **Fourier no es ninguna magia negra de análisis avanzado**: es **exactamente el mismo Álgebra Lineal**, pero aplicada a funciones periódicas y a vectores muestreados en una computadora.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Configuracion estetica de graficos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 11
print("Librerias cargadas correctamente.")


---
## 1. Repaso de Álgebra Lineal: Vectores, Producto Interno y Proyección

> ⚠️ **Aclaración importante de vocabulario:**
> A veces en clase o en apuntes rápidos se dice por error *"producto cruz"*.
> En Álgebra Lineal:
> - El **producto cruz** (o vectorial $\vec{u} \times \vec{v}$) **solo existe en $\mathbb{R}^3$** y genera otro vector perpendicular.
> - La operación que mide **proyección, similitud, ángulo y solapamiento** entre dos elementos es el **producto interno** (o producto escalar / *dot product*):
>   $$\langle \vec{u}, \vec{v} \rangle = \vec{u} \cdot \vec{v} = \sum_{i=1}^n u_i v_i$$
>
> Es este **producto interno** el que generalizamos a funciones y el que está en el corazón de Fourier.

### ¿Qué hace el producto interno geométricamente?
1. **Mide parecido / alineación:**
   - Si dos vectores apuntan en la misma dirección, su producto interno es positivo y grande.
   - Si son **perpendiculares (ortogonales)**, $\langle \vec{u}, \vec{v} \rangle = 0$.
2. **Extrae coordenadas (Proyección):**
   Si tenemos una base ortonormal canónica en $\mathbb{R}^3$, por ejemplo $\hat{e}_1 = [1, 0, 0]^T$, y un vector cualquiera $\vec{v} = [v_1, v_2, v_3]^T$:
   $$c_1 = \vec{v} \cdot \hat{e}_1 = v_1 \cdot 1 + v_2 \cdot 0 + v_3 \cdot 0 = v_1$$
   **El producto interno actúa como un "sensor"**: cuando hacés el producto interno de $\vec{v}$ contra un vector de la base $\hat{e}_i$, el producto anula todas las demás componentes y **pesca únicamente la coordenada en esa dirección**.


In [ ]:
# Demostracion en codigo: Proyeccion ortogonal en R^2
v = np.array([4.0, 3.0])      # Vector original v
e1 = np.array([1.0, 0.0])     # Base canonica x
e2 = np.array([0.0, 1.0])     # Base canonica y

# Coordenadas mediante producto interno
c1 = np.dot(v, e1)  # v_x
c2 = np.dot(v, e2)  # v_y

print(f"Vector v: {v}")
print(f"Coordenada c1 (proyeccion sobre e1): {c1}")
print(f"Coordenada c2 (proyeccion sobre e2): {c2}")

# Reconstruccion: v = c1*e1 + c2*e2
v_reconstruido = c1 * e1 + c2 * e2
print(f"Vector reconstruido: {v_reconstruido}")

# Grafico
fig, ax = plt.subplots(figsize=(6, 5))
ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1, color='tab:blue', label=r'Vector $\vec{v} = [4, 3]$')
ax.quiver(0, 0, e1[0], e1[1], angles='xy', scale_units='xy', scale=1, color='tab:green', label=r'$\hat{e}_1$')
ax.quiver(0, 0, e2[0], e2[1], angles='xy', scale_units='xy', scale=1, color='tab:red', label=r'$\hat{e}_2$')
ax.plot([v[0], v[0]], [0, v[1]], 'k--', alpha=0.5)
ax.plot([0, v[0]], [v[1], v[1]], 'k--', alpha=0.5)
ax.set_xlim(-1, 5)
ax.set_ylim(-1, 4)
ax.set_aspect('equal')
ax.set_title("El producto interno 'pesca' la coordenada en cada eje")
ax.legend()
plt.show()


---
## 2. El Espacio de Funciones: Vectores de Infinitas Dimensiones

### ¿Qué tiene que ver una función con un vector? ¡Son lo mismo!
Pensá en cómo guardás un sonido o una señal de temperatura:
- Muestreo cada 1 segundo: tenés una lista de números $[f(1), f(2), \dots, f(N)]$. ¡Esto es un vector en $\mathbb{R}^N$!
- Muestreo cada 1 milisegundo: el vector tiene 1000 veces más componentes.
- Si hacés el espaciado infinitesimal ($dx \to 0$), la lista ya no tiene índices discretos $i \in \{1, 2, \dots, N\}$, sino un índice continuo $x$.

Una función continua $f(x)$ **es simplemente un vector con infinitas componentes continuas**.

### ¿Cumple los axiomas de espacio vectorial?
Absolutamente sí:
1. **Suma:** $(f + g)(x) = f(x) + g(x)$ (sumás componente a componente).
2. **Multiplicación por escalar:** $(c \cdot f)(x) = c \cdot f(x)$.
3. Cumple distributividad, asociatividad, elemento neutro (la función nula $0$), etc.

### ¿Cómo se calcula el producto interno en este espacio?
En $\mathbb{R}^N$:
$$\langle \vec{u}, \vec{v} \rangle = \sum_{i=1}^N u_i v_i$$

Al pasar a un índice continuo $x$, la sumatoria $\sum$ se convierte de forma natural en una **integral** $\int$:
$$\langle f, g \rangle = \int_a^b f(x) g(x) dx$$

¡Exactamente la misma idea! Multiplicás las funciones punto a punto y sumás todas las componentes. Si $\langle f, g \rangle = 0$, decimos que las dos funciones son **ortogonales**.


In [ ]:
# Demostracion: De la sumatoria discreta a la integral continua
x_fino = np.linspace(-np.pi, np.pi, 1000)
f_cont = np.sin(x_fino)
g_cont = np.cos(x_fino)

# Aproximacion por sumatoria de Riemann (discreta)
dx = x_fino[1] - x_fino[0]
producto_interno_aprox = np.sum(f_cont * g_cont) * dx

# Integral teorica: integral_{-pi}^{pi} sin(x)*cos(x) dx = 0
print(f"Puntos usados: {len(x_fino)}")
print(f"Producto interno sum(f * g) * dx: {producto_interno_aprox:.10e} (practicamente 0)")


---
## 3. Convolución y Correlación: El Producto Interno con Desplazamiento

Si el producto interno $\langle f, g \rangle = \int f(x) g(x) dx$ mide qué tanto se parecen dos funciones cuando están perfectamente superpuestas...

### ¿Qué pasa si una señal está corrida en el tiempo?
Supongamos que tenés una señal $f(t)$ y querés buscar un patrón $g(t)$, pero no sabés en qué instante aparece.
Lo natural es **deslizar** $g$ a lo largo del tiempo por un desplazamiento $\tau$ e ir calculando el producto interno en cada punto:
$$(f \star g)(t) = \int_{-\infty}^\infty f(\tau) g(t + \tau) d\tau \quad \text{(Correlación)}$$

Y si además invertís el tiempo de la segunda función:
$$(f * g)(t) = \int_{-\infty}^\infty f(\tau) g(t - \tau) d\tau \quad \text{(Convolución)}$$

> **La Convolución es un "escáner de producto interno":**
> En cada instante $t$, la convolución calcula el producto interno entre $f$ y la versión desplazada e invertida de $g$. Si en algún instante las dos formas coinciden, el producto interno produce un pico máximo.


In [ ]:
# Demostracion visual: Deteccion de patrones mediante correlacion (producto interno desplazado)
t = np.linspace(0, 10, 1000)
# Senal ruidosa con un pulso en t=6
patron = np.exp(-((t[:100] - 0.5)**2) / 0.05)  # Pulso gaussiano de prueba
senal = np.random.normal(0, 0.2, len(t))
senal[600:700] += patron * 2.0  # Insertamos el patron en t=6

# Calculamos la correlacion cruzada (producto interno deslizante)
correlacion = np.correlate(senal, patron, mode='same')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.plot(t, senal, color='gray', label='Senal ruidosa')
ax1.plot(t[600:700], senal[600:700], color='tab:red', label='Patron oculto en t ≈ 6')
ax1.set_title("Senal en el dominio del tiempo")
ax1.legend()

ax2.plot(t, correlacion, color='tab:blue', lw=2, label='Producto interno desplazado (Correlacion)')
ax2.axvline(6.0, color='tab:red', linestyle='--', label='Pico maximo de alineacion')
ax2.set_title("La correlacion da un pico gigante cuando las senales encajan")
ax2.set_xlabel("Tiempo t")
ax2.legend()
plt.tight_layout()
plt.show()


---
## 4. Senos y Cosenos como "Vectores Canónicos" (La Base Ortogonal)

En $\mathbb{R}^3$, la base canónica es $\{\hat{i}, \hat{j}, \hat{k}\}$.
¿Qué tiene de especial esa base?
$$\hat{i} \cdot \hat{j} = 0, \quad \hat{i} \cdot \hat{k} = 0, \quad \hat{j} \cdot \hat{k} = 0$$
$$\hat{i} \cdot \hat{i} = 1, \quad \hat{j} \cdot \hat{j} = 1, \quad \hat{k} \cdot \hat{k} = 1$$
Son **perpendiculares entre sí** (ortogonales) y de longitud 1 (ortonormales).

En el espacio de funciones periódicas en $[-\pi, \pi]$, Fourier propuso usar como base:
$$\mathcal{B} = \{ 1, \cos(x), \sin(x), \cos(2x), \sin(2x), \dots, \cos(nx), \sin(nx), \dots \}$$

### Demostración: ¿Son realmente ortogonales? ("Da 0 si son distintas, da $\pi$ si son iguales")

Calculemos el producto interno entre dos funciones de esta base en el intervalo $[-\pi, \pi]$:

#### Caso 1: Coseno con Coseno de distinta frecuencia ($n \neq m$)
Usando la identidad trigonométrica $\cos(A)\cos(B) = \frac{1}{2}[\cos(A-B) + \cos(A+B)]$:
$$\langle \cos(nx), \cos(mx) \rangle = \int_{-\pi}^\pi \cos(nx) \cos(mx) dx = \frac{1}{2} \int_{-\pi}^\pi [\cos((n-m)x) + \cos((n+m)x)] dx$$
Como $n \neq m$, ambas son ondas cosenoidales completas en el intervalo $[-\pi, \pi]$. La integral de un coseno completo es su área positiva cancelándose exactamente con su área negativa:
$$= \frac{1}{2} \left[ \frac{\sin((n-m)x)}{n-m} + \frac{\sin((n+m)x)}{n+m} \right]_{-\pi}^\pi = 0$$

#### Caso 2: Coseno con Coseno de la misma frecuencia ($n = m$)
$$\langle \cos(nx), \cos(nx) \rangle = \int_{-\pi}^\pi \cos^2(nx) dx = \int_{-\pi}^\pi \frac{1 + \cos(2nx)}{2} dx = \left[ \frac{x}{2} + \frac{\sin(2nx)}{4n} \right]_{-\pi}^\pi = \frac{\pi - (-\pi)}{2} = \pi$$

#### Caso 3: Coseno con Seno (cruzados)
Para cualquier $n$ y $m$:
$$\langle \cos(nx), \sin(mx) \rangle = \int_{-\pi}^\pi \underbrace{\cos(nx)}_{\text{par}} \cdot \underbrace{\sin(mx)}_{\text{impar}} dx = \int_{-\pi}^\pi \text{función impar } dx = 0$$

### 📌 Resumen de Ortogonalidad:
$$\int_{-\pi}^\pi \cos(nx) \cos(mx) dx = \begin{cases} 0 & \text{si } n \neq m \\ \pi & \text{si } n = m \end{cases}$$
$$\int_{-\pi}^\pi \sin(nx) \sin(mx) dx = \begin{cases} 0 & \text{si } n \neq m \\ \pi & \text{si } n = m \end{cases}$$
$$\int_{-\pi}^\pi \cos(nx) \sin(mx) dx = 0 \quad \text{siempre}$$

> 💡 **¡Son exactamente como los vectores canónicos $\hat{i}, \hat{j}, \hat{k}$!**
> Cada frecuencia $n$ apunta en una dimensión completamente perpendicular a todas las demás frecuencias.


In [ ]:
# Demostracion numerica: Matriz de Productos Internos entre Frecuencias
N_frecuencias = 5
matriz_cosenos = np.zeros((N_frecuencias, N_frecuencias))
x = np.linspace(-np.pi, np.pi, 2000)
dx = x[1] - x[0]

for n in range(1, N_frecuencias + 1):
    for m in range(1, N_frecuencias + 1):
        cos_n = np.cos(n * x)
        cos_m = np.cos(m * x)
        # Producto interno: integral aproximada
        matriz_cosenos[n-1, m-1] = np.sum(cos_n * cos_m) * dx

fig, ax = plt.subplots(figsize=(6, 5))
cax = ax.matshow(matriz_cosenos, cmap='Blues')
fig.colorbar(cax)
ax.set_xticks(range(N_frecuencias))
ax.set_yticks(range(N_frecuencias))
ax.set_xticklabels([f"cos({i}x)" for i in range(1, N_frecuencias + 1)])
ax.set_yticklabels([f"cos({i}x)" for i in range(1, N_frecuencias + 1)])
ax.set_title(r"Matriz de $\langle \cos(nx), \cos(mx) \rangle$ en $[-\pi, \pi]$", pad=20)

for i in range(N_frecuencias):
    for j in range(N_frecuencias):
        val = matriz_cosenos[i, j]
        ax.text(j, i, f"{val:.2f}" if abs(val) > 0.05 else "0.00",
                ha="center", va="center", color="black" if val < 2 else "white", fontweight='bold')

plt.show()
print(f"Valor teorico en la diagonal: pi = {np.pi:.4f}")


---
## 5. El Pasaje a Coeficientes $a_n$ y $b_n$: Desmitificando "Las Cosas Feas"

Ahora que sabemos que los senos y cosenos son una base ortogonal, escribamos cualquier función $f(x)$ como **combinación lineal** de los vectores de esa base:

$$f(x) = \frac{a_0}{2} + \sum_{n=1}^\infty \Big( a_n \cos(nx) + b_n \sin(nx) \Big)$$

En los libros de texto, te presentan de golpe estas dos fórmulas intimidantes:
$$a_k = \frac{1}{\pi} \int_{-\pi}^\pi f(x) \cos(kx) dx \quad (\text{cosa fea 1})$$
$$b_k = \frac{1}{\pi} \int_{-\pi}^\pi f(x) \sin(kx) dx \quad (\text{cosa fea 2})$$

### ¿De dónde salen? ¡De una proyección ortogonal elemental de Álgebra Lineal!

Recordá la fórmula de Álgebra Lineal para proyectar un vector $\vec{v}$ sobre una base ortogonal $\{\vec{e}_1, \vec{e}_2, \dots\}$:
$$c_k = \frac{\langle \vec{v}, \vec{e}_k \rangle}{\langle \vec{e}_k, \vec{e}_k \rangle} = \frac{\langle \vec{v}, \vec{e}_k \rangle}{\|\vec{e}_k\|^2}$$

Hagamos **exactamente lo mismo**: tomemos la ecuación de $f(x)$ y hagamos el **producto interno a ambos lados con $\cos(kx)$**:

$$\int_{-\pi}^\pi f(x) \cos(kx) dx = \int_{-\pi}^\pi \left( \frac{a_0}{2} + \sum_{n=1}^\infty [a_n \cos(nx) + b_n \sin(nx)] \right) \cos(kx) dx$$

Distribuimos la integral término a término:
- $\int \frac{a_0}{2} \cos(kx) dx = 0$ (porque la constante 1 es ortogonal a $\cos(kx)$).
- Todos los $\int b_n \sin(nx) \cos(kx) dx = 0$ (porque senos y cosenos son ortogonales entre sí).
- En la sumatoria de $\int a_n \cos(nx) \cos(kx) dx$, **todos los términos dan $0$ excepto cuando $n = k$**:

$$\int_{-\pi}^\pi f(x) \cos(kx) dx = a_k \int_{-\pi}^\pi \cos^2(kx) dx$$

Y como demostramos recién, $\int_{-\pi}^\pi \cos^2(kx) dx = \pi$. Por lo tanto:
$$\int_{-\pi}^\pi f(x) \cos(kx) dx = a_k \cdot \pi$$

Despejás $a_k$:
$$\mathbf{a_k = \frac{1}{\pi} \int_{-\pi}^\pi f(x) \cos(kx) dx}$$

Y si hacés el producto interno con $\sin(kx)$, sobrevive únicamente el término $b_k$:
$$\mathbf{b_k = \frac{1}{\pi} \int_{-\pi}^\pi f(x) \sin(kx) dx}$$

> 🎉 **¡No hay ninguna fórmula fea ni misteriosa!**
> Los coeficientes $a_k$ y $b_k$ son simplemente las **coordenadas** del vector $f(x)$ proyectado sobre los ejes $\cos(kx)$ y $\sin(kx)$.


In [ ]:
# Demostracion practica: Descomposicion y reconstruccion de una onda cuadrada
# f(x) = 1 si x >= 0, -1 si x < 0
x = np.linspace(-np.pi, np.pi, 2000)
f_cuadrada = np.where(x >= 0, 1.0, -1.0)

# Como la onda cuadrada es impar:
# a_n = 0 para todo n (los cosenos son pares, producto interno par*impar = 0)
# b_n = (4 / (n * pi)) para n impar, y 0 para n par
def calcular_reconstruccion(K_max):
    reconstruccion = np.zeros_like(x)
    for n in range(1, K_max + 1, 2):  # solo n impares
        b_n = 4.0 / (n * np.pi)
        reconstruccion += b_n * np.sin(n * x)
    return reconstruccion

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, f_cuadrada, 'k--', lw=2, label=r'Onda cuadrada original $f(x)$')

for K, col in zip([1, 3, 9, 29], ['tab:orange', 'tab:green', 'tab:blue', 'tab:red']):
    ax.plot(x, calcular_reconstruccion(K), label=f'Suma con {K} armonicos', lw=1.5)

ax.set_title("Reconstruccion sumando coordenadas en la base de Fourier")
ax.set_xlabel("x")
ax.legend(loc='upper left')
plt.show()


---
## 6. La Elegancia de Euler: $e^{ix} = \cos(x) + i\sin(x)$

Tener dos juegos de coeficientes ($a_n$ para cosenos y $b_n$ para senos) es engorroso.
Aquí entra la identidad más hermosa de la matemática:
$$\mathbf{e^{i\theta} = \cos(\theta) + i\sin(\theta)}$$

### ¿Por qué esto simplifica todo?
En vez de dos bases trigonométricas reales separadas, usamos una **única base exponencial compleja**:
$$\mathcal{B}_{compleja} = \{ \dots, e^{-i2x}, e^{-ix}, 1, e^{ix}, e^{i2x}, \dots \}$$

Cualquier función se escribe como una única serie:
$$\mathbf{f(x) = \sum_{n=-\infty}^\infty c_n e^{inx}}$$

¿Y cómo calculamos la coordenada compleja $c_n$?
Nuevamente con el **producto interno** (en espacios complejos usamos el conjugado $\langle f, g \rangle = \int f(x) \overline{g(x)} dx$):
$$\mathbf{c_n = \frac{1}{2\pi} \int_{-\pi}^\pi f(x) e^{-inx} dx}$$

### ¿Qué representa geométricamente el número complejo $c_n$?
Cada $c_n$ es un vector en el plano complejo:
- **Su magnitud $|c_n|$:** es la **amplitud** (cuánta energía o peso tiene esa frecuencia).
- **Su ángulo $\angle c_n$:** es la **fase** (cuánto está desplazada horizontalmente esa onda).

$$c_n = |c_n| e^{i \phi_n} = \underbrace{|c_n|}_{\text{Amplitud}} \angle \underbrace{\phi_n}_{\text{Fase}}$$


In [ ]:
# Visualizacion geometrica de los coeficientes de Fourier en el plano complejo
n_vals = np.arange(1, 10)
# Para la onda cuadrada: c_n = -2j / (n * pi) para n impar
c_n_vals = np.zeros(len(n_vals), dtype=complex)
for i, n in enumerate(n_vals):
    if n % 2 != 0:
        c_n_vals[i] = -2j / (n * np.pi)

amplitudes = np.abs(c_n_vals)
fases = np.angle(c_n_vals)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.stem(n_vals, amplitudes, basefmt=" ")
ax1.set_title("Espectro de Magnitud $|c_n|$ (Amplitud)")
ax1.set_xlabel("Frecuencia n")
ax1.set_ylabel("Magnitud")

ax2.stem(n_vals, np.degrees(fases), basefmt=" ", linefmt='tab:red', markerfmt='ro')
ax2.set_title(r"Espectro de Fase $\angle c_n$ (en grados)")
ax2.set_xlabel("Frecuencia n")
ax2.set_ylabel("Fase (grados)")
plt.tight_layout()
plt.show()


---
## 7. La Forma Discreta: Transformada Discreta de Fourier (DFT)

En una computadora, una señal de audio o una fila de una imagen no tiene infinitos puntos continuos; tiene $N$ muestras discretas:
$$\vec{x} = [x_0, x_1, \dots, x_{N-1}]^T \in \mathbb{C}^N$$

### La DFT es literalmente una multiplicación matriz-vector:
$$\mathbf{\vec{X} = W \vec{x}}$$

Donde $W$ es la célebre **Matriz de Fourier** de tamaño $N \times N$:
$$W_{kn} = e^{-i \frac{2\pi}{N} kn}$$

$$\begin{bmatrix} X_0 \\ X_1 \\ X_2 \\ \vdots \\ X_{N-1} \end{bmatrix} =
\begin{bmatrix}
1 & 1 & 1 & \dots & 1 \\
1 & W_N^1 & W_N^2 & \dots & W_N^{N-1} \\
1 & W_N^2 & W_N^4 & \dots & W_N^{2(N-1)} \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
1 & W_N^{N-1} & W_N^{2(N-1)} & \dots & W_N^{(N-1)(N-1)}
\end{bmatrix}
\begin{bmatrix} x_0 \\ x_1 \\ x_2 \\ \vdots \\ x_{N-1} \end{bmatrix}$$

donde $W_N = e^{-i \frac{2\pi}{N}}$.

### ¿Qué propiedades tiene la matriz $W$?
1. Sus columnas son **ortogonales** entre sí en $\mathbb{C}^N$.
2. Por lo tanto, su inversa es simplemente su **traspuesta conjugada** (dividida por $N$):
   $$\mathbf{W^{-1} = \frac{1}{N} W^H}$$
3. Reconstrucción (IDFT):
   $$\vec{x} = \frac{1}{N} W^H \vec{X}$$

> 🏆 **Conclusión definitiva:**
> La Transformada Discreta de Fourier (DFT) es **exactamente un cambio de base ortogonal** de Álgebra Lineal:
> - Base canónica del tiempo/espacio: $\{\hat{e}_0, \hat{e}_1, \dots, \hat{e}_{N-1}\}$ (valores en cada pixel o instante).
> - Base de Fourier: las columnas de la matriz $W^H$ (ondas sinusoidales discretas de frecuencias crecientes).


In [ ]:
# Demostracion en codigo: Construir la matriz de Fourier W y multiplicar a mano
N = 8  # Tamano pequeno para visualizar la matriz
n = np.arange(N)
k = n.reshape((N, 1))

# Matriz de Fourier W
W = np.exp(-2j * np.pi * k * n / N)

# Senal de prueba aleatoria
x_prueba = np.array([1.0, 3.0, 5.0, 2.0, -1.0, 0.0, 4.0, 2.0])

# 1. DFT mediante multiplicacion matricial pura: X = W @ x
X_matricial = W @ x_prueba

# 2. DFT mediante la funcion estandar de NumPy (FFT)
X_numpy = np.fft.fft(x_prueba)

# Comparamos resultados
print("Resultados de la multiplicacion matricial W @ x:")
print(np.round(X_matricial, 4))
print("\nResultados de np.fft.fft(x):")
print(np.round(X_numpy, 4))

error_maximo = np.max(np.abs(X_matricial - X_numpy))
print(f"\nDiferencia maxima entre ambos metodos: {error_maximo:.2e} (Son identicos)")

# 3. Inversa mediante W_conjugada_transpuesta / N
x_recuperado = (W.conj().T @ X_matricial) / N
print(f"Se recupera la senal original con W.H / N: {np.allclose(x_prueba, x_recuperado)}")


In [ ]:
# Visualizacion de las columnas de la matriz de Fourier (la base ortogonal discreta)
fig, axes = plt.subplots(4, 2, figsize=(10, 8), sharex=True)
for k_idx in range(4):
    # Fila k de la matriz: parte real (coseno) y parte imaginaria (-seno)
    base_real = np.real(W[k_idx, :])
    base_imag = np.imag(W[k_idx, :])
    
    axes[k_idx, 0].stem(n, base_real, basefmt=" ")
    axes[k_idx, 0].set_ylabel(f"k = {k_idx}")
    if k_idx == 0:
        axes[k_idx, 0].set_title("Parte Real (Cosenos discretos)")
        
    axes[k_idx, 1].stem(n, base_imag, basefmt=" ", linefmt='tab:orange', markerfmt='tab:orange')
    if k_idx == 0:
        axes[k_idx, 1].set_title("Parte Imaginaria (Senos discretos)")

axes[-1, 0].set_xlabel("Indice de muestra n")
axes[-1, 1].set_xlabel("Indice de muestra n")
plt.suptitle("Las filas de la Matriz de Fourier son ondas discretas de frecuencias crecientes", y=1.02)
plt.tight_layout()
plt.show()


---
## 8. Conexión con Procesamiento de Imágenes (Fourier 2D)

En una imagen digital $f(x, y)$, tenemos dos dimensiones espaciales:
1. Las funciones base ya no son ondas de 1D, sino **ondas planas 2D** que tienen:
   - Una **frecuencia horizontal** $u$.
   - Una **frecuencia vertical** $v$.
   - Una **orientación o dirección** $\theta = \arctan(v / u)$.
2. La DFT 2D es separable: aplicás la matriz de Fourier $W$ a las filas y luego a las columnas:
   $$\mathbf{F = W \cdot f \cdot W^T}$$
3. **Interpretación física en imágenes:**
   - **Centro del espectro (bajas frecuencias):** Zonas lisas, fondos, iluminación general y sombras suaves.
   - **Periferia del espectro (altas frecuencias):** Bordes nítidos, texturas finas, detalles pequeños y ruido.

---
### 🏁 Síntesis final
| Concepto en Álgebra Lineal | Concepto en Series de Fourier | Concepto en DFT / Imágenes |
| :--- | :--- | :--- |
| Vector $\vec{v} \in \mathbb{R}^N$ | Función continua $f(x)$ | Matriz de píxeles $f(x, y)$ |
| Producto Interno $\sum u_i v_i$ | Integral $\int f(x) g(x) dx$ | Suma matricial $\sum_{x,y} f(x,y) e^{-i...}$ |
| Base ortonormal $\{\hat{e}_i\}$ | Base ortogonal $\{\cos(nx), \sin(nx)\}$ | Columnas de la matriz $W^H$ |
| Coordenada $c_i = \vec{v} \cdot \hat{e}_i$ | Coeficientes $a_n, b_n, c_n$ | Coeficientes DFT $F(u, v)$ |
| Cambio de base $[v]_B = P^{-1} [v]$ | Reconstrucción $\sum c_n e^{inx}$ | Multiplicación $F = W f W^T$ |
